# 2. Limpieza, normalización y unión

Punto 3 del enunciado. Justificad cada corrección, eliminación o imputación y anotadla en `docs/decisiones.md`.

**Responsables:** Alex (Adzuna y unión) y Vahab (Tecnoempleo).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / 'src'))
from config import RAW_ADZUNA, RAW_TECNOEMPLEO, PROCESSED, REFERENCE

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 2.1 API *(Vahab)*

_Pendiente de la API que se elija._

In [ ]:
# TODO (Vahab)

## 2.2 Tecnoempleo *(Alex)*

La lógica está en `src/limpieza_tecnoempleo.py`; aquí se ejecuta paso a paso y se comprueba cada resultado. Los datos originales (`data/raw/`) no se modifican.

In [ ]:
import limpieza_tecnoempleo as lt

raw = lt.cargar_raw()
print(raw.shape)
raw.head(3)

### Paso 1. Duplicados

Las ofertas 100 % remotas aparecen en el listado de Madrid **y** en el de Barcelona. No son errores, pero contarlas dos veces inflaría los resultados. Nos quedamos con una fila por `id_oferta` y guardamos en qué listados apareció.

In [ ]:
print('Filas originales:', len(raw), '· ofertas distintas:', raw['id_oferta'].nunique())
repetidas = raw[raw.duplicated('id_oferta', keep=False)]
print('Modalidad de las repetidas:', repetidas.drop_duplicates('id_oferta')['modalidad'].value_counts().to_dict())

### Pasos 2 a 5. Limpieza completa

- **Modalidad:** Presencial, Híbrido, Remoto o No indicada.
- **Ciudad:** Madrid, Barcelona, Otra ciudad (p. ej. Almería, que aparece en el listado de Madrid) o Sin ciudad (remoto).
- **Salario:** de texto a números en € brutos/año. Mensual × 12, por hora × 1.760. Un "mensual" ≥ 10.000 € se trata como anual (`salario_corregido`).
- **Fechas:** hay ofertas publicadas entre 2017 y 2025 que siguen activas; no se borran, se marca `reciente` (≤ 90 días).
- **Jornada parcial:** salarios muy bajos de helpdesk de fin de semana o jornada parcial. No son errores; se marcan en `posible_jornada_parcial`.
- **Tecnologías:** se buscan las 33 del estudio en las etiquetas y el título, por palabra completa ("java" ≠ "javascript", "git" ≠ "github").

In [ ]:
te_ofertas, te_tecnologias = lt.limpiar(raw)
lt.resumen(te_ofertas, te_tecnologias)

### Comprobaciones

In [ ]:
# Salarios convertidos desde mensual o por hora
te_ofertas.loc[te_ofertas['salario_unidad'].isin(['mensual', 'hora']),
              ['titulo', 'salario_texto', 'salario_min', 'salario_max', 'salario_corregido']]

In [ ]:
# Valores extremos del salario (se revisan en el EDA, no se eliminan aquí)
cols = ['titulo', 'salario_texto', 'salario_medio', 'posible_jornada_parcial']
display(te_ofertas.nsmallest(5, 'salario_medio')[cols])
display(te_ofertas.nlargest(5, 'salario_medio')[cols])

In [ ]:
# Ciudad × modalidad
pd.crosstab(te_ofertas['ciudad'], te_ofertas['modalidad'], margins=True)

In [ ]:
# Tecnologías sin ninguna oferta
sin_ofertas = set(lt.cargar_alias()) - set(te_tecnologias['tecnologia'])
print('Sin ofertas en Tecnoempleo:', sorted(sin_ofertas))

In [ ]:
# Nulos que quedan (son ausencias reales de información, no errores)
te_ofertas.isna().sum()[lambda s: s > 0]

In [ ]:
te_ofertas.to_csv(PROCESSED / 'tecnoempleo_ofertas.csv', index=False)
te_tecnologias.to_csv(PROCESSED / 'tecnoempleo_tecnologias.csv', index=False)

## 2.3 Unión de las fuentes

1. Esquema común y columna `fuente`.
2. Posibles ofertas repetidas entre fuentes (título + empresa + ciudad).
3. Tabla agregada tecnología × ciudad.

In [ ]:
# TODO (Alex)
# ofertas.to_csv(PROCESSED / 'ofertas.csv', index=False)
# agregada.to_csv(PROCESSED / 'agregada_tecnologia_ciudad.csv', index=False)